# Imports

In [6]:
import pandas as pd
import pickle
import os
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from transformers import AutoTokenizer, AutoModel

# Load Data

In [4]:
parent_dir = os.path.dirname(os.getcwd())
data_dir = os.path.join(parent_dir, 'data')
print(f"Data directory: {data_dir}")
data = pd.read_json(os.path.join(data_dir, 'dataset_cleaned.json'), lines=True)
data.head()

Data directory: c:\git_clones\tech_challenge_illuin_technology\data


,prob_desc_sample_outputs,prob_desc_notes,prob_desc_description,prob_desc_output_spec,prob_desc_input_spec,difficulty,prob_desc_sample_inputs,source_code,tags
0,"[""2\n2 4\n3 3\n3 1""]",NaN,"Numbers $$$1, 2, 3, \dots n$$$ (each integer f...","For each test case, in the first line, print t...",The first line contains one integer $$$t$$$ ($...,1000.0,"[""1\n4""]",\ndef ii(): return int(input())\ndef mi(): ret...,[math]
1,"[""4\n10\n4\n0""]","NoteIn the first test case of the example, the...","There are $$$n$$$ positive integers $$$a_1, a_...",For $$$t$$$ test cases print the answers in th...,The first line of the input contains one integ...,1200.0,"[""4\n6\n40 6 40 3 20 1\n1\n1024\n4\n2 4 8 16\n...",a = int(input())\nfor i in range(a):\n f = ...,[number theory]
2,"[""5"", ""16"", ""18""]",NoteIn the first example it is possible to con...,You are given an undirected graph consisting o...,Print one integer — the minimum number of coin...,The first line contains two integers $$$n$$$ a...,1900.0,"[""3 2\n1 3 3\n2 3 5\n2 1 1"", ""4 0\n1 3 3 7"", ""...",def read_nums():\n return [int(x) for x in ...,[graphs]
3,"[""2\n5000 9\n1\n7 \n4\n800 70 6 9000 \n1\n1000...",NaN,A positive (strictly greater than zero) intege...,Print $$$t$$$ answers to the test cases. Each ...,The first line contains an integer $$$t$$$ ($$...,800.0,"[""5\n5009\n7\n9876\n10000\n10""]",t = int(input())\nfor i in range(t):\n canP...,[math]
4,"[""1"", ""0""]",NoteThe first test case corresponds to the tre...,You are given a tree with $$$n$$$ vertices. Yo...,Print a single integer — the minimum number o...,The first line contains an integer $$$n$$$ ($$...,2800.0,"[""6\n4 5\n2 6\n3 2\n1 2\n2 4"", ""4\n2 4\n4 1\n3...",import sys\nfrom collections import defaultdic...,"[graphs, trees]"


In [9]:
data.shape

(2678, 9)

# Data Scaling

In [5]:
df=data.copy()

In [7]:
scaler = StandardScaler()
df['difficulty'] = scaler.fit_transform(df[['difficulty']])

In [8]:
df.head()

,prob_desc_sample_outputs,prob_desc_notes,prob_desc_description,prob_desc_output_spec,prob_desc_input_spec,difficulty,prob_desc_sample_inputs,source_code,tags
0,"[""2\n2 4\n3 3\n3 1""]",NaN,"Numbers $$$1, 2, 3, \dots n$$$ (each integer f...","For each test case, in the first line, print t...",The first line contains one integer $$$t$$$ ($...,-1.267887,"[""1\n4""]",\ndef ii(): return int(input())\ndef mi(): ret...,[math]
1,"[""4\n10\n4\n0""]","NoteIn the first test case of the example, the...","There are $$$n$$$ positive integers $$$a_1, a_...",For $$$t$$$ test cases print the answers in th...,The first line of the input contains one integ...,-0.923753,"[""4\n6\n40 6 40 3 20 1\n1\n1024\n4\n2 4 8 16\n...",a = int(input())\nfor i in range(a):\n f = ...,[number theory]
2,"[""5"", ""16"", ""18""]",NoteIn the first example it is possible to con...,You are given an undirected graph consisting o...,Print one integer — the minimum number of coin...,The first line contains two integers $$$n$$$ a...,0.280714,"[""3 2\n1 3 3\n2 3 5\n2 1 1"", ""4 0\n1 3 3 7"", ""...",def read_nums():\n return [int(x) for x in ...,[graphs]
3,"[""2\n5000 9\n1\n7 \n4\n800 70 6 9000 \n1\n1000...",NaN,A positive (strictly greater than zero) intege...,Print $$$t$$$ answers to the test cases. Each ...,The first line contains an integer $$$t$$$ ($$...,-1.612020,"[""5\n5009\n7\n9876\n10000\n10""]",t = int(input())\nfor i in range(t):\n canP...,[math]
4,"[""1"", ""0""]",NoteThe first test case corresponds to the tre...,You are given a tree with $$$n$$$ vertices. Yo...,Print a single integer — the minimum number o...,The first line contains an integer $$$n$$$ ($$...,1.829316,"[""6\n4 5\n2 6\n3 2\n1 2\n2 4"", ""4\n2 4\n4 1\n3...",import sys\nfrom collections import defaultdic...,"[graphs, trees]"


# Embedding

In [12]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
embedding_model = AutoModel.from_pretrained("microsoft/codebert-base")

Loading weights: 100%|██████████| 199/199 [00:00<?, ?it/s]


Si tu veux utiliser CodeBERT comme dans la phase de pré-entraînement (concaténation texte + code), tu peux effectivement concaténer une description (texte) et du code source :

In [ ]:
# input_text = description + tokenizer.sep_token + code_source

* [CLS] is a special token in
front of the two segments, whose final hidden representation is considered as the aggregated sequence representation for classification or ranking.
* The output of CodeBERT includes :
    * contextualvector representation of each token, for both natural language and code.
    * the representation of [CLS], which works as the aggregated sequence representation.

In [10]:
def embed(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    #tokens = tokenizer.tokenize(text)
    #token_ids = inputs["input_ids"][0].tolist()
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.squeeze(0)
    return embeddings[0].numpy()  # [CLS] embedding

## Embedding prob_desc_description

In [13]:
df['prob_desc_description_embedding']=df['prob_desc_description'].apply(lambda x: embed(x, tokenizer, embedding_model))

In [15]:
df.head()

,prob_desc_sample_outputs,prob_desc_notes,prob_desc_description,prob_desc_output_spec,prob_desc_input_spec,difficulty,prob_desc_sample_inputs,source_code,tags,prob_desc_description_embedding
0,"[""2\n2 4\n3 3\n3 1""]",NaN,"Numbers $$$1, 2, 3, \dots n$$$ (each integer f...","For each test case, in the first line, print t...",The first line contains one integer $$$t$$$ ($...,-1.267887,"[""1\n4""]",\ndef ii(): return int(input())\ndef mi(): ret...,[math],"[0.14105268, 0.34100538, -0.22524361, 0.139201..."
1,"[""4\n10\n4\n0""]","NoteIn the first test case of the example, the...","There are $$$n$$$ positive integers $$$a_1, a_...",For $$$t$$$ test cases print the answers in th...,The first line of the input contains one integ...,-0.923753,"[""4\n6\n40 6 40 3 20 1\n1\n1024\n4\n2 4 8 16\n...",a = int(input())\nfor i in range(a):\n f = ...,[number theory],"[0.100468084, 0.3743114, -0.23292993, 0.159436..."
2,"[""5"", ""16"", ""18""]",NoteIn the first example it is possible to con...,You are given an undirected graph consisting o...,Print one integer — the minimum number of coin...,The first line contains two integers $$$n$$$ a...,0.280714,"[""3 2\n1 3 3\n2 3 5\n2 1 1"", ""4 0\n1 3 3 7"", ""...",def read_nums():\n return [int(x) for x in ...,[graphs],"[0.08530584, 0.42754978, -0.25779152, 0.094448..."
3,"[""2\n5000 9\n1\n7 \n4\n800 70 6 9000 \n1\n1000...",NaN,A positive (strictly greater than zero) intege...,Print $$$t$$$ answers to the test cases. Each ...,The first line contains an integer $$$t$$$ ($$...,-1.612020,"[""5\n5009\n7\n9876\n10000\n10""]",t = int(input())\nfor i in range(t):\n canP...,[math],"[0.21549368, 0.3164608, -0.24520843, 0.0788967..."
4,"[""1"", ""0""]",NoteThe first test case corresponds to the tre...,You are given a tree with $$$n$$$ vertices. Yo...,Print a single integer — the minimum number o...,The first line contains an integer $$$n$$$ ($$...,1.829316,"[""6\n4 5\n2 6\n3 2\n1 2\n2 4"", ""4\n2 4\n4 1\n3...",import sys\nfrom collections import defaultdic...,"[graphs, trees]","[0.08409342, 0.42758143, -0.27324587, 0.234087..."


## Embedding prob_desc_input_spec

In [26]:
df['prob_desc_input_spec_combined'] = (df['prob_desc_input_spec'].fillna("").astype(str) + " | Example: " + df['prob_desc_sample_inputs'].fillna("").astype(str))                         

In [27]:
df['prob_desc_input_spec_combined_embedding'] = df['prob_desc_input_spec_combined'].apply(lambda x: embed(x, tokenizer, embedding_model))

## Embedding prob_desc_output_spec

In [28]:
df['prob_desc_output_spec_combined'] = (df['prob_desc_output_spec'].fillna("").astype(str) + " | Example: " + df['prob_desc_sample_outputs'].fillna("").astype(str))

In [29]:
df['prob_desc_output_spec_combined_embedding']=df['prob_desc_output_spec_combined'].apply(lambda x: embed(x, tokenizer, embedding_model))

## Embedding source_code

In [18]:
df['source_code_embedding']=df['source_code'].apply(lambda x: embed(x, tokenizer, embedding_model))

## Embedding prob_desc_notes

In [25]:
df['prob_desc_notes'] = df['prob_desc_notes'].fillna("")
df['prob_desc_notes_embedding']=df['prob_desc_notes'].apply(lambda x: embed(x, tokenizer, embedding_model))

# Label Encoding

In [34]:
mlb = MultiLabelBinarizer()
df['tags_encoded'] = list(mlb.fit_transform(df['tags']))

In [35]:
df['tag_combination'] = df['tags'].apply(lambda x: tuple(sorted(x)))
le = LabelEncoder()
df['tag_combination_encoded'] = le.fit_transform(df['tag_combination'])

# Save Encoded Data

In [40]:
import pickle
with open(os.path.join(data_dir, "df_encoded.pkl"), "wb") as f:
    pickle.dump(df, f)